In [1]:
using Julog, Test

In [2]:
# Test composition and transitive relations using the traditional Zen lineage
clauses = @julog [
    ancestor(sakyamuni, bodhidharma) <<= true,
    teacher(bodhidharma, huike) <<= true,
    teacher(huike, sengcan) <<= true,
    teacher(sengcan, daoxin) <<= true,
    teacher(daoxin, hongren) <<= true,
    teacher(hongren, huineng) <<= true,
    ancestor(A, B) <<= teacher(A, B),
    ancestor(A, C) <<= teacher(B, C) & ancestor(A, B),
    grandteacher(A, C) <<= teacher(A, B) & teacher(B, C)
];

In [3]:
# Is Sakyamuni the dharma ancestor of Huineng?
goals = @julog [ancestor(sakyamuni, huineng)]
sat, subst = resolve(goals, clauses);
# @test sat == true

(true, Dict{Var, Term}[{}])

In [4]:

# Who are the grandteachers of whom?
goals = @julog [grandteacher(X, Y)]
sat, subst = resolve(goals, clauses)
subst = Set(subst)

Set{Dict{Var, Term}} with 4 elements:
  {Y => hongren, X => sengcan}
  {Y => huineng, X => daoxin}
  {Y => sengcan, X => bodhidharma}
  {Y => daoxin, X => huike}

In [5]:
@test @varsub({X => bodhidharma, Y => sengcan}) in subst
@test @varsub({X => huike, Y => daoxin}) in subst
@test @varsub({X => sengcan, Y => hongren}) in subst
@test @varsub({X => daoxin, Y => huineng}) in subst

Test Passed

In [6]:

# Test that forward chaining produces the same / correct results
fwd_sat, fwd_subst = derive(goals, clauses)
fwd_subst = Set(fwd_subst)

Set{Dict{Var, Term}} with 4 elements:
  {Y => hongren, X => sengcan}
  {Y => huineng, X => daoxin}
  {Y => sengcan, X => bodhidharma}
  {Y => daoxin, X => huike}

In [7]:
@test fwd_sat == sat
@test fwd_subst == subst
n_init = 6
n_ancestor = 5 + 15
n_grandteacher = 4
@test length(derivations(clauses, Inf)) == n_ancestor + n_grandteacher + n_init

Test Passed

In [8]:

# Test clause table manipulation
table = index_clauses(clauses[1:4])

Dict{Symbol, Dict{Symbol, Vector{Clause}}} with 2 entries:
  :teacher  => Dict(:__all__=>[teacher(bodhidharma, huike), teacher(huike, seng…
  :ancestor => Dict(:__all__=>[ancestor(sakyamuni, bodhidharma)], :sakyamuni=>[…

In [9]:
table = insert_clauses!(table, clauses[4:end])

Dict{Symbol, Dict{Symbol, Vector{Clause}}} with 3 entries:
  :teacher      => Dict(:__all__=>[teacher(bodhidharma, huike), teacher(huike, …
  :grandteacher => Dict(:__var__=>[grandteacher(A, C) <<= teacher(A, B) & teach…
  :ancestor     => Dict(:__var__=>[ancestor(A, B) <<= teacher(A, B), ancestor(A…

In [10]:
@test table == index_clauses(clauses)

Test Passed

In [11]:
subtract_clauses!(table, clauses[5:end])

Dict{Symbol, Dict{Symbol, Vector{Clause}}} with 3 entries:
  :teacher      => Dict(:__all__=>[teacher(bodhidharma, huike), teacher(huike, …
  :grandteacher => Dict(:__var__=>[], :__all__=>[])
  :ancestor     => Dict(:__var__=>[], :__all__=>[ancestor(sakyamuni, bodhidharm…

In [12]:
@test Set(deindex_clauses(table)) == Set(clauses[1:4])

Test Passed